# Simple linear BRDF — reflectance / radiance vs SZA and cos(SZA)

For a single **patch** and a single **wavelength**, this notebook pulls every
QC-passed acquisition of that patch and plots four scatter panels:

| | vs SZA | vs cos(SZA) |
|---|---|---|
| **Reflectance** | panel 1 | panel 3 |
| **Radiance**    | panel 2 | panel 4 |

Each panel gets **two** regression lines — ordinary least squares (OLS) and the
robust Theil–Sen slope (the repo's drift-analysis convention) — with the fit
statistics printed in the top-right corner.

**Why cos(SZA)?** In the whiteboard BRDF model (see `brdf_exploration.ipynb`) the
surface directional response is the linear *bracket* $c + d\cos\theta$, so a
straight line through reflectance-vs-$\cos\theta$ recovers the site's BRDF
sensitivity ($d/c$ = slope/intercept). Radiance carries the same geometry factor
times the raw solar/instrument signal, shown alongside for reference.

**One point per acquisition.** SZA is essentially constant across a 10 km patch
within a single overpass, so the regression signal lives *across scenes*: each
point is the patch-**median** reflectance/radiance at the chosen wavelength for
one acquisition, plotted against that scene's mean SZA. Points are colored by
acquisition year, so any time/geometry confounding is visible by eye.

**Inputs consumed** (all written by the extraction step; no network here):
`data/geometry/geometry_{PICS}_{PATCH}.csv`,
`data/reflectance/{PICS}_{PATCH}_{ts}_perchannel_rfl.csv`,
`data/radiance/{PICS}_{PATCH}_{ts}_perchannel_rad.csv`, and the QC worksheet
`data/qc/{PICS}_{PATCH}_qc.csv`.


## 1. Configuration

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ------------------------------------------------------------------
# USER-CONFIGURABLE INPUTS
# ------------------------------------------------------------------
PICS_ID       = "Libya_4"     # underscored site id
PATCH_ID      = "10_7_1"      # patch_id within the PICS
WAVELENGTH_NM = 660.0         # target wavelength (nm); nearest EMIT channel is used

Y_STAT   = "median"           # patch statistic for the y-value: median | mean | ...
SZA_STAT = "sza_mean"         # per-scene SZA column from the geometry table
APPLY_QC = True               # drop keep=FALSE scenes (falls back to all if no sheet)

# ------------------------------------------------------------------
# Repo-relative paths (walk up until we find the package dir).
# ------------------------------------------------------------------
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "emit_pics").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / "data"
GEOM_PATH = DATA_ROOT / "geometry" / f"geometry_{PICS_ID}_{PATCH_ID}.csv"
RFL_DIR   = DATA_ROOT / "reflectance"
RAD_DIR   = DATA_ROOT / "radiance"
QC_PATH   = DATA_ROOT / "qc" / f"{PICS_ID}_{PATCH_ID}_qc.csv"
FIG_DIR   = REPO_ROOT / "figures" / "BRDF"

RFL_LABEL = "Reflectance (unitless)"
RAD_LABEL = "Radiance (µW cm⁻² sr⁻¹ nm⁻¹)"

plt.rcParams["figure.dpi"] = 110
print(f"Repo root:  {REPO_ROOT}")
print(f"Patch:      {PICS_ID} / {PATCH_ID}   @ {WAVELENGTH_NM:.1f} nm ({Y_STAT})")
print(f"Geometry:   {GEOM_PATH}   exists={GEOM_PATH.exists()}")

## 2. Load the scenes and apply the QC gate

Scenes come from the per-scene geometry table. `apply_qc_filter` is called with
`require_review=False`, so it drops any `keep=FALSE` scenes if a reviewed
worksheet exists and otherwise passes every scene through — it never blocks the
notebook.

In [ ]:
from emit_pics.qc_gate import apply_qc_filter

if not GEOM_PATH.exists():
    raise FileNotFoundError(
        f"Geometry table not found:\n  {GEOM_PATH}\n"
        "Run the extraction step (run_single_patch_cube_extract.py or "
        "run_PICS_allpatches_cube_extract.py) for this patch first."
    )

scenes = pd.read_csv(GEOM_PATH).sort_values("datetime").reset_index(drop=True)
n_all = len(scenes)

if APPLY_QC:
    scenes, qc_report = apply_qc_filter(scenes, QC_PATH, require_review=False,
                                        timestamp_col="timestamp")
    print(f"QC: {qc_report['n_before']} -> {qc_report['n_after']} scenes "
          f"({qc_report['n_excluded']} excluded {qc_report['reasons']})")
    if not QC_PATH.exists():
        print(f"  (no QC worksheet at {QC_PATH.name}; using all scenes)")
else:
    print("QC filtering disabled (APPLY_QC=False).")

print(f"{len(scenes)} of {n_all} scene(s) will be plotted.")
scenes[["timestamp", "datetime", "has_rfl", SZA_STAT]].head()

## 3. Assemble one (SZA, reflectance, radiance) point per scene

For each surviving scene we read its per-channel reflectance and radiance CSVs
and pick the channel nearest the target wavelength (reflectance and radiance have
slightly different band grids, so each is matched independently). RAD-only scenes
contribute a radiance point with reflectance left as NaN.

In [ ]:
def nearest_channel(csv_path: Path, target_nm: float, stat_col: str):
    """(actual_wl, value) at the channel nearest *target_nm*, or (nan, nan)."""
    if not csv_path.exists():
        return np.nan, np.nan
    d = pd.read_csv(csv_path)
    if stat_col not in d.columns or "central_wl" not in d.columns:
        return np.nan, np.nan
    i = (d["central_wl"] - target_nm).abs().idxmin()
    return float(d["central_wl"].iloc[i]), float(d[stat_col].iloc[i])


rows = []
rfl_wls, rad_wls = [], []
for _, s in scenes.iterrows():
    ts = str(s["timestamp"])
    rfl_wl, rfl = nearest_channel(RFL_DIR / f"{PICS_ID}_{PATCH_ID}_{ts}_perchannel_rfl.csv",
                                  WAVELENGTH_NM, f"{Y_STAT}_rfl")
    rad_wl, rad = nearest_channel(RAD_DIR / f"{PICS_ID}_{PATCH_ID}_{ts}_perchannel_rad.csv",
                                  WAVELENGTH_NM, f"{Y_STAT}_rad")
    if np.isfinite(rfl_wl): rfl_wls.append(rfl_wl)
    if np.isfinite(rad_wl): rad_wls.append(rad_wl)
    rows.append({"timestamp": ts, "datetime": s["datetime"],
                 "sza": float(s[SZA_STAT]), "rfl": rfl, "rad": rad})

df = pd.DataFrame(rows)
df["cos_sza"] = np.cos(np.radians(df["sza"]))
_dt = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
df["year"] = _dt.dt.year + (_dt.dt.dayofyear - 1) / 365.25

# Report the actual channels used (should be one value each).
rfl_ch = f"{np.median(rfl_wls):.1f} nm" if rfl_wls else "n/a"
rad_ch = f"{np.median(rad_wls):.1f} nm" if rad_wls else "n/a"
print(f"Target {WAVELENGTH_NM:.1f} nm  ->  reflectance channel {rfl_ch}, radiance channel {rad_ch}")
print(f"{df['rfl'].notna().sum()} reflectance point(s), {df['rad'].notna().sum()} radiance point(s), "
      f"SZA range {df['sza'].min():.1f}–{df['sza'].max():.1f}°")
df

## 4. Regression helper (OLS + Theil–Sen)

`fit_stats` returns both fits on the finite (x, y) pairs. `plot_panel` draws the
scatter (colored by year), overlays both regression lines, and annotates the
top-right corner with slope / R² / p for OLS and slope / Kendall-τ / p for
Theil–Sen, plus the point count.

In [ ]:
def fit_stats(x, y):
    """OLS + Theil–Sen fit over finite pairs. Returns a dict (n always present)."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    out = {"n": int(x.size)}
    if x.size < 3 or np.ptp(x) == 0:
        return out                         # too few points / no x-spread to fit
    ols = stats.linregress(x, y)
    ts = stats.theilslopes(y, x)           # (slope, intercept, lo_slope, hi_slope)
    tau, tau_p = stats.kendalltau(x, y)
    out.update(ols_slope=ols.slope, ols_intercept=ols.intercept,
               ols_r2=ols.rvalue ** 2, ols_p=ols.pvalue,
               ts_slope=ts[0], ts_intercept=ts[1], ts_lo=ts[2], ts_hi=ts[3],
               ts_tau=tau, ts_p=tau_p)
    return out


def plot_panel(ax, x, y, xlabel, ylabel, title, color_by):
    fit = fit_stats(x, y)
    sc = ax.scatter(x, y, c=color_by, cmap="viridis", s=30,
                    edgecolor="k", linewidth=0.3, zorder=3)
    if "ols_slope" in fit:
        xs = np.array([np.nanmin(x), np.nanmax(x)])
        ax.plot(xs, fit["ols_intercept"] + fit["ols_slope"] * xs,
                "-", color="#d62728", lw=1.7, label="OLS", zorder=2)
        ax.plot(xs, fit["ts_intercept"] + fit["ts_slope"] * xs,
                "--", color="#1f77b4", lw=1.7, label="Theil–Sen", zorder=2)
        txt = (f"OLS  m={fit['ols_slope']:+.4g}\n"
               f"     R²={fit['ols_r2']:.3f}  p={fit['ols_p']:.2g}\n"
               f"T–S  m={fit['ts_slope']:+.4g}\n"
               f"     τ={fit['ts_tau']:+.2f}  p={fit['ts_p']:.2g}\n"
               f"n = {fit['n']}")
        ax.legend(loc="lower left", fontsize=7, framealpha=0.75)
    else:
        txt = f"n = {fit['n']}\n(insufficient x-spread\nto fit)"
    ax.text(0.97, 0.97, txt, transform=ax.transAxes, ha="right", va="top",
            fontsize=8, family="monospace", zorder=4,
            bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.grid(alpha=0.25)
    return fit, sc

## 5. The four panels

Row 1 plots reflectance and radiance side by side against **SZA**; row 2 plots the
same two against **cos(SZA)**. The shared colorbar encodes acquisition year.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9.5))

_, sc = plot_panel(axes[0, 0], df["sza"], df["rfl"],
                   "SZA (degrees)", RFL_LABEL, "Reflectance vs SZA", df["year"])
plot_panel(axes[0, 1], df["sza"], df["rad"],
           "SZA (degrees)", RAD_LABEL, "Radiance vs SZA", df["year"])
plot_panel(axes[1, 0], df["cos_sza"], df["rfl"],
           "cos(SZA)", RFL_LABEL, "Reflectance vs cos(SZA)", df["year"])
plot_panel(axes[1, 1], df["cos_sza"], df["rad"],
           "cos(SZA)", RAD_LABEL, "Radiance vs cos(SZA)", df["year"])

fig.suptitle(f"{PICS_ID}  patch {PATCH_ID}   —   "
             f"{WAVELENGTH_NM:.0f} nm ({Y_STAT} over patch, {len(df)} scenes)",
             fontsize=13, y=0.995)
fig.tight_layout(rect=(0, 0, 0.93, 0.98))
cax = fig.add_axes((0.945, 0.12, 0.015, 0.76))
fig.colorbar(sc, cax=cax, label="acquisition year")

FIG_DIR.mkdir(parents=True, exist_ok=True)
out_png = FIG_DIR / f"simple_linear_BRDF_{PICS_ID}_{PATCH_ID}_{WAVELENGTH_NM:.0f}nm.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
print(f"Saved {out_png}")
plt.show()

## 6. Fit summary table

The BRDF-relevant number is the **cos(SZA)** slope: for reflectance, slope /
intercept is the site's fractional directional sensitivity ($d/c$). OLS and
Theil–Sen slopes are tabulated for all four panels.

In [ ]:
summary = []
for xcol, xname in [("sza", "SZA"), ("cos_sza", "cos(SZA)")]:
    for ycol, yname in [("rfl", "reflectance"), ("rad", "radiance")]:
        f = fit_stats(df[xcol], df[ycol])
        row = {"y": yname, "x": xname, "n": f["n"]}
        if "ols_slope" in f:
            row.update({"ols_slope": f["ols_slope"], "ols_r2": f["ols_r2"],
                        "ols_p": f["ols_p"], "ts_slope": f["ts_slope"],
                        "ts_p": f["ts_p"]})
            if xcol == "cos_sza" and f["ols_intercept"] != 0:
                row["ols_slope/intercept"] = f["ols_slope"] / f["ols_intercept"]
        summary.append(row)

pd.DataFrame(summary)

## 7. Pre-flight sanity checks for the joint BRDF+drift fit

Before trusting the four-parameter joint regression (the `a + b·t`, `c + d·cosθ`
whiteboard system in `brdf_exploration.ipynb`), the checks below probe whether
this patch's acquisitions actually support it. Each cell ends with a printed
`PASS` / `CHECK` / `FAIL` line plus the governing number, so the section reads as
a checklist. All use the existing per-acquisition `df`
(`[timestamp, datetime, sza, cos_sza, rfl, rad, year]`, QC already applied),
patch-median y-values, and degrade to reflectance-only if radiance is absent.

### 7.1  Time–geometry separability

If SZA and acquisition time are correlated, the joint fit cannot cleanly separate
the geometry coefficient `d` from the drift coefficient `b`. This scatter and its
Pearson `corr(t, SZA)` gate everything downstream: `|r| < 0.3` → well conditioned;
higher → expect inflated standard error on the drift coefficient.

In [ ]:
# --- shared setup for the 7.x checks (defined once, reused below) ---
MEAN_RFL = float(np.nanmean(df["rfl"]))
MEAN_RAD = float(np.nanmean(df["rad"])) if df["rad"].notna().any() else np.nan
HAS_RAD  = int(df["rad"].notna().sum()) >= 3

def _resid_cos(ycol):
    # OLS residuals of ycol vs cos(SZA) over finite pairs -> (sub_df, resid, fit).
    x = df["cos_sza"].to_numpy(float); y = df[ycol].to_numpy(float)
    m = np.isfinite(x) & np.isfinite(y)
    fit = fit_stats(df["cos_sza"], df[ycol])
    if "ols_slope" not in fit:
        return df[m].reset_index(drop=True), np.array([]), fit
    r = y[m] - (fit["ols_intercept"] + fit["ols_slope"] * x[m])
    return df[m].reset_index(drop=True), r, fit

def _pct(v, base):
    return 100.0 * v / base if (base and np.isfinite(base)) else np.nan

# --- 7.1 ---
xt = df["year"].to_numpy(float); ys = df["sza"].to_numpy(float)
m = np.isfinite(xt) & np.isfinite(ys)
r_ts, p_ts = (stats.pearsonr(xt[m], ys[m]) if m.sum() >= 3 else (np.nan, np.nan))

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.scatter(xt[m], ys[m], s=32, edgecolor="k", linewidth=0.3)
ax.set_xlabel("acquisition time (decimal year)"); ax.set_ylabel("SZA (degrees)")
ax.set_title("7.1  Time–geometry separability")
ax.text(0.97, 0.03, f"r(t, SZA) = {r_ts:+.3f}\np = {p_ts:.2g}\nn = {int(m.sum())}",
        transform=ax.transAxes, ha="right", va="bottom", family="monospace",
        fontsize=9, bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
ax.grid(alpha=0.25); plt.tight_layout(); plt.show()

_v = "PASS" if abs(r_ts) < 0.3 else ("CHECK" if abs(r_ts) < 0.7 else "FAIL")
print(f"[7.1] {_v}: |corr(t, SZA)| = {abs(r_ts):.3f} "
      f"({'well conditioned' if _v == 'PASS' else 'drift/geometry entangled — inflated SE on drift coef'})")

### 7.2  Drift preview via residuals

A crude two-stage drift estimate: strip the geometry trend (residuals of the
signal-vs-cos(SZA) fit), then regress those residuals on time. The slope over the
mean signal is a rough %/yr to sanity-check against the eventual joint-fit
`b/a`. It is **not** the real estimate — it ignores the t–SZA covariance from 7.1
— just a preview.

In [ ]:
def _drift_preview(ycol, base, ax, label):
    sub, r, fit = _resid_cos(ycol)
    if r.size < 3:
        ax.set_title(f"{label}: n/a")
        ax.text(0.5, 0.5, "insufficient data", transform=ax.transAxes,
                ha="center", va="center")
        return np.nan, np.nan
    lr = stats.linregress(sub["year"].to_numpy(float), r)
    pct_yr = _pct(lr.slope, base)
    ax.scatter(sub["year"], r, s=28, edgecolor="k", linewidth=0.3)
    xs = np.array([sub["year"].min(), sub["year"].max()])
    ax.plot(xs, lr.intercept + lr.slope * xs, "-", color="#d62728", lw=1.6)
    ax.axhline(0.0, color="0.6", lw=0.8, ls=":")
    ax.set_xlabel("decimal year")
    ax.set_title(f"{label}  {pct_yr:+.3f}%/yr (p={lr.pvalue:.2g})")
    ax.grid(alpha=0.25)
    return pct_yr, lr.pvalue

ncol = 2 if HAS_RAD else 1
fig, axes = plt.subplots(1, ncol, figsize=(6.5 * ncol, 4.2), squeeze=False)
axes = axes[0]
axes[0].set_ylabel("residual (signal units)")
pct_r, p_r = _drift_preview("rfl", MEAN_RFL, axes[0], "reflectance")
if HAS_RAD:
    pct_d, p_d = _drift_preview("rad", MEAN_RAD, axes[1], "radiance")
else:
    pct_d, p_d = np.nan, np.nan
    print("note: no radiance points — reflectance-only drift preview.")
plt.tight_layout(); plt.show()

_flag = (np.isfinite(p_r) and p_r < 0.05) or (np.isfinite(p_d) and p_d < 0.05)
_v = "CHECK" if _flag else "PASS"
_radtxt = f", rad={pct_d:+.3f}%/yr" if HAS_RAD else ""
print(f"[7.2] {_v}: crude drift rfl={pct_r:+.3f}%/yr{_radtxt} — compare against joint-fit b/a")

### 7.3  Bracket linearity

The model assumes the directional response is linear in cos(SZA): `c + d·cosθ`.
If the residuals of that linear fit show significant curvature against cos(SZA), a
quadratic (or physical BRDF) term is needed before the joint fit is trustworthy.
We fit a quadratic to the reflectance residuals and test its leading coefficient.

In [ ]:
def _poly_pvals(x, y, degree=2):
    # OLS polynomial fit; returns (coeffs_high_to_low, pvalues_high_to_low).
    X = np.vander(x, degree + 1)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    dof = len(y) - X.shape[1]
    if dof <= 0:
        return beta, np.full(beta.shape, np.nan)
    sigma2 = float(resid @ resid) / dof
    cov = sigma2 * np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    return beta, 2.0 * stats.t.sf(np.abs(beta / se), dof)

sub, r, fit = _resid_cos("rfl")
if r.size >= 4:
    cx = sub["cos_sza"].to_numpy(float)
    beta, pvals = _poly_pvals(cx, r, 2)
    q2, q2_p = beta[0], pvals[0]
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.scatter(cx, r, s=28, edgecolor="k", linewidth=0.3)
    gx = np.linspace(cx.min(), cx.max(), 100)
    ax.plot(gx, np.polyval(beta, gx), "-", color="#d62728", lw=1.6, label="quadratic")
    ax.axhline(0.0, color="0.6", lw=0.8, ls=":")
    ax.set_xlabel("cos(SZA)"); ax.set_ylabel("reflectance residual")
    ax.set_title("7.3  Bracket linearity (residuals vs cos SZA)")
    ax.legend(fontsize=8); ax.grid(alpha=0.25); plt.tight_layout(); plt.show()
    _v = "PASS" if (not np.isfinite(q2_p) or q2_p >= 0.05) else "FAIL"
    print(f"[7.3] {_v}: quadratic coef = {q2:+.4g} (p={q2_p:.2g}) — "
          f"{'linear bracket adequate' if _v == 'PASS' else 'significant curvature; add higher-order term'}")
else:
    print("[7.3] CHECK: insufficient reflectance points to test bracket curvature")

### 7.4  Regressor choice — θ vs cos(θ)

Is cos(SZA) really the better linear regressor than raw SZA? We compare the
residual patterns of both reflectance fits against SZA, on a shared y-axis. The
more patternless (lower-RMSE, structure-free) panel wins — cos(SZA) is expected
to. RMSEs are reported as % of mean reflectance.

In [ ]:
def _resid_vs_sza(xcol, ycol):
    x = df[xcol].to_numpy(float); y = df[ycol].to_numpy(float)
    m = np.isfinite(x) & np.isfinite(y)
    fit = fit_stats(df[xcol], df[ycol])
    if "ols_slope" not in fit:
        return df["sza"].to_numpy(float)[m], np.array([])
    r = y[m] - (fit["ols_intercept"] + fit["ols_slope"] * x[m])
    return df["sza"].to_numpy(float)[m], r

sza_x, r_sza = _resid_vs_sza("sza", "rfl")
cos_x, r_cos = _resid_vs_sza("cos_sza", "rfl")
if r_sza.size >= 3 and r_cos.size >= 3:
    rmse_sza = float(np.sqrt(np.mean(r_sza ** 2)))
    rmse_cos = float(np.sqrt(np.mean(r_cos ** 2)))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    axes[0].scatter(sza_x, r_sza, s=26, edgecolor="k", linewidth=0.3)
    axes[0].set_title(f"resid of SZA fit  (RMSE {_pct(rmse_sza, MEAN_RFL):.3f}%)")
    axes[1].scatter(cos_x, r_cos, s=26, edgecolor="k", linewidth=0.3)
    axes[1].set_title(f"resid of cos(SZA) fit  (RMSE {_pct(rmse_cos, MEAN_RFL):.3f}%)")
    for a in axes:
        a.axhline(0.0, color="0.6", lw=0.8, ls=":")
        a.set_xlabel("SZA (degrees)"); a.grid(alpha=0.25)
    axes[0].set_ylabel("reflectance residual")
    plt.tight_layout(); plt.show()
    _v = "PASS" if rmse_cos <= rmse_sza else "CHECK"
    print(f"[7.4] {_v}: RMSE cos(SZA)={_pct(rmse_cos, MEAN_RFL):.3f}% vs "
          f"SZA={_pct(rmse_sza, MEAN_RFL):.3f}% — "
          f"{'cos(SZA) preferred' if _v == 'PASS' else 'raw SZA as good or better; revisit regressor'}")
else:
    print("[7.4] CHECK: insufficient reflectance points to compare regressors")

### 7.5  Outlier gate — OLS vs Theil–Sen

If a few scenes drag the least-squares slope, closed-form OLS in the joint fit is
risky. We check whether the OLS slope of the reflectance-vs-cos(SZA) fit falls
inside the Theil–Sen slope CI (`ts_lo`, `ts_hi` from `fit_stats`). Inside →
outliers are not distorting, OLS is safe. Outside → the three largest `|residual|`
scenes are flagged for inspection before considering an L1 estimator.

In [ ]:
fit = fit_stats(df["cos_sza"], df["rfl"])
if "ols_slope" in fit:
    inside = fit["ts_lo"] <= fit["ols_slope"] <= fit["ts_hi"]
    print(f"OLS slope       = {fit['ols_slope']:+.4g}")
    print(f"Theil-Sen slope = {fit['ts_slope']:+.4g}  CI [{fit['ts_lo']:+.4g}, {fit['ts_hi']:+.4g}]")
    if inside:
        _v = "PASS"
    else:
        _v = "CHECK"
        sub, r, _ = _resid_cos("rfl")
        worst = sub.assign(absres=np.abs(r)).nlargest(3, "absres")
        print("  3 largest |residual| scenes to inspect:")
        for _, w in worst.iterrows():
            print(f"    {w['timestamp']}  SZA={w['sza']:.1f}  |resid|={w['absres']:.4g}")
    print(f"[7.5] {_v}: OLS slope {'inside' if inside else 'OUTSIDE'} Theil-Sen CI — "
          f"{'outliers not distorting; OLS safe' if inside else 'robust (L1) estimator advised'}")
else:
    print("[7.5] CHECK: insufficient reflectance points for the outlier gate")

### 7.6  Sampling adequacy

Even with low global t–SZA correlation, `d` is only well constrained if each
geometry bin is sampled across a range of dates. We bin SZA at 5° and, for each
occupied bin, annotate the scene count and the time span (max − min decimal year)
of its scenes. Bins holding ≥2 scenes with <1 yr span constrain `d` only within a
narrow epoch — locally confounding it with drift.

In [ ]:
sza_all = df["sza"].to_numpy(float); yr_all = df["year"].to_numpy(float)
m = np.isfinite(sza_all) & np.isfinite(yr_all)
sza_all, yr_all = sza_all[m], yr_all[m]
if sza_all.size:
    lo = np.floor(sza_all.min() / 5) * 5; hi = np.ceil(sza_all.max() / 5) * 5
    edges = np.arange(lo, hi + 5, 5)
    fig, ax = plt.subplots(figsize=(7.5, 4.4))
    ax.hist(sza_all, bins=edges, edgecolor="k", color="#6baed6")
    narrow = 0
    for i in range(len(edges) - 1):
        sel = (sza_all >= edges[i]) & (sza_all < edges[i + 1])
        n_i = int(sel.sum())
        if n_i == 0:
            continue
        span = float(yr_all[sel].max() - yr_all[sel].min())
        if n_i >= 2 and span < 1.0:
            narrow += 1
        ax.text((edges[i] + edges[i + 1]) / 2, n_i, f"n={n_i}\nΔt={span:.1f}y",
                ha="center", va="bottom", fontsize=7)
    ax.set_xlabel("SZA (degrees)"); ax.set_ylabel("scene count")
    ax.set_title("7.6  Geometry sampling vs epoch"); ax.margins(y=0.15)
    plt.tight_layout(); plt.show()
    _v = "PASS" if narrow == 0 else "CHECK"
    print(f"[7.6] {_v}: {narrow} bin(s) with >=2 scenes span <1 yr — "
          f"{'geometry sampled across epochs' if narrow == 0 else 'narrow-epoch bins locally confound d with drift'}")
else:
    print("[7.6] CHECK: no SZA samples")

### 7.7  Seasonal residual check

A residual signal locked to day-of-year (not SZA) points to a seasonal effect the
SZA term cannot absorb — relative azimuth, aerosol climatology, BRDF azimuth
dependence. We fit a first harmonic (`A·sin + B·cos` of `2π·doy/365.25`) to the
reflectance-vs-cos(SZA) residuals and F-test it against the flat model. Surviving
seasonality becomes structured noise the joint fit inherits.

In [ ]:
sub, r, fit = _resid_cos("rfl")
if r.size >= 5:
    doy = pd.to_datetime(sub["datetime"], utc=True, errors="coerce").dt.dayofyear.to_numpy(float)
    w = 2.0 * np.pi * doy / 365.25
    X = np.column_stack([np.sin(w), np.cos(w), np.ones_like(w)])
    beta, *_ = np.linalg.lstsq(X, r, rcond=None)
    fitted = X @ beta
    amp = float(np.hypot(beta[0], beta[1]))
    rss_full = float(np.sum((r - fitted) ** 2))
    rss_flat = float(np.sum((r - r.mean()) ** 2))
    n = r.size
    if rss_full > 0 and n > 3:
        F = ((rss_flat - rss_full) / 2.0) / (rss_full / (n - 3))
        p_seas = float(stats.f.sf(F, 2, n - 3))
    else:
        p_seas = np.nan
    order = np.argsort(doy)
    fig, ax = plt.subplots(figsize=(6.8, 4.2))
    ax.scatter(doy, r, s=28, edgecolor="k", linewidth=0.3)
    ax.plot(doy[order], fitted[order], "-", color="#d62728", lw=1.6, label="1st harmonic")
    ax.axhline(0.0, color="0.6", lw=0.8, ls=":")
    ax.set_xlabel("day of year"); ax.set_ylabel("reflectance residual")
    ax.set_title("7.7  Seasonal residual check"); ax.legend(fontsize=8); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show()
    _v = "PASS" if (not np.isfinite(p_seas) or p_seas >= 0.05) else "CHECK"
    print(f"[7.7] {_v}: seasonal amplitude = {_pct(amp, MEAN_RFL):.3f}% of mean "
          f"(F-test p={p_seas:.2g}) — "
          f"{'no significant seasonality' if _v == 'PASS' else 'seasonal structure beyond SZA'}")
else:
    print("[7.7] CHECK: insufficient reflectance points for a seasonal fit")

### 7.8  Spectral coherence of d/c

The fractional directional sensitivity `d/c` (slope/intercept of the cos(SZA)
fit) should vary smoothly with wavelength. Isolated spikes at channels bordering
the 940/1140/1380/1880 nm water-vapor features are the known ~0.087 nm
wavelength-shift artifact masquerading as directional sensitivity — not real
BRDF. We sweep channels both away from and adjacent to those features (reusing
`nearest_channel`) and plot `d/c` with Theil–Sen error bars.

In [ ]:
def _dc_at(target):
    vals = []
    for _, s in scenes.iterrows():
        ts = str(s["timestamp"])
        _, v = nearest_channel(RFL_DIR / f"{PICS_ID}_{PATCH_ID}_{ts}_perchannel_rfl.csv",
                               target, f"{Y_STAT}_rfl")
        vals.append(v)
    f = fit_stats(df["cos_sza"], pd.Series(vals))
    if "ols_slope" not in f or f["ols_intercept"] == 0:
        return np.nan, np.nan
    dc = f["ols_slope"] / f["ols_intercept"]
    yerr = ((f["ts_hi"] - f["ts_lo"]) / 2.0) / abs(f["ols_intercept"])
    return dc, yerr

WV = [940, 1140, 1380, 1880]   # water-vapor feature centers
TARGETS = [550, 660, 780, 865, 900, 960, 1070, 1250, 1360, 1400, 1600, 1860, 2100, 2200]
dc = np.array([_dc_at(t) for t in TARGETS], float)
vals, errs = dc[:, 0], dc[:, 1]
ok = np.isfinite(vals)
if ok.sum() >= 3:
    tt = np.array(TARGETS, float)
    med = np.median(vals[ok]); mad = np.median(np.abs(vals[ok] - med)) or np.nan
    thresh = 3.5 * 1.4826 * mad if (np.isfinite(mad) and mad > 0) else np.inf
    spike = ok & (np.abs(vals - med) > thresh)
    fig, ax = plt.subplots(figsize=(8, 4.4))
    ax.errorbar(tt[ok], vals[ok], yerr=errs[ok], fmt="o", ms=5, capsize=3,
                color="#1f77b4", ecolor="0.6", lw=1)
    if spike.any():
        ax.scatter(tt[spike], vals[spike], s=90, facecolors="none",
                   edgecolors="#d62728", linewidths=1.6, label="spike")
        ax.legend(fontsize=8)
    for wv in WV:
        ax.axvline(wv, color="0.75", ls="--", lw=0.9)
    ax.set_xlabel("wavelength (nm)"); ax.set_ylabel("d/c  (slope / intercept)")
    ax.set_title("7.8  Spectral coherence of d/c"); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show()
    n_sp = int(spike.sum())
    _v = "PASS" if n_sp == 0 else "CHECK"
    _spk = "" if n_sp == 0 else " @ " + ", ".join(f"{int(t)}nm" for t in tt[spike])
    print(f"[7.8] {_v}: {n_sp} d/c spike(s) beyond 3.5*MAD{_spk} — "
          f"{'smooth spectral variation' if n_sp == 0 else 'inspect near water-vapor edges (wavelength-shift artifact)'}")
else:
    print("[7.8] CHECK: insufficient reflectance data for the spectral d/c sweep")